In [1]:
from google.colab import files
files.upload()  # 上傳 kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"allenwang2005","key":"c4c0a182a8a888f0597b0fb8f765d5dc"}'}

In [2]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [3]:
!kaggle competitions download -c 2025-information-retrieval-extraction-homework-2

 99% 4.22G/4.27G [00:54<00:01, 31.8MB/s]
100% 4.27G/4.27G [00:54<00:00, 84.0MB/s]


In [4]:
!unzip *.zip -d data

串流輸出內容已截斷至最後 5000 行。
  inflating: data/train_papers_latex/train_papers_latex/2211.13868v2/eps/ppt/slideLayouts/_rels/slideLayout10.xml.rels  
  inflating: data/train_papers_latex/train_papers_latex/2211.13868v2/eps/ppt/slideLayouts/_rels/slideLayout11.xml.rels  
  inflating: data/train_papers_latex/train_papers_latex/2211.13868v2/eps/ppt/slideLayouts/_rels/slideLayout2.xml.rels  
  inflating: data/train_papers_latex/train_papers_latex/2211.13868v2/eps/ppt/slideLayouts/_rels/slideLayout3.xml.rels  
  inflating: data/train_papers_latex/train_papers_latex/2211.13868v2/eps/ppt/slideLayouts/_rels/slideLayout4.xml.rels  
  inflating: data/train_papers_latex/train_papers_latex/2211.13868v2/eps/ppt/slideLayouts/_rels/slideLayout5.xml.rels  
  inflating: data/train_papers_latex/train_papers_latex/2211.13868v2/eps/ppt/slideLayouts/_rels/slideLayout6.xml.rels  
  inflating: data/train_papers_latex/train_papers_latex/2211.13868v2/eps/ppt/slideLayouts/_rels/slideLayout7.xml.rels  
  inflating: data

In [ ]:
import pandas as pd

data = pd.read_json('data/train.jsonl', lines=True)

### Test SentenceTransformer

In [ ]:
import tqdm
data['image_caption_label'] = data.groupby('paper_id')['image_caption'].transform(lambda x: ','.join(x))

from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('intfloat/e5-large-v2')

for index, row in tqdm.tqdm(data.iterrows(), total=len(data)):
    query = row['query']
    caption_label = row['image_caption_label']

    captions = caption_label.split(',')

    query_embedding = model.encode(query, convert_to_tensor=True)
    caption_embeddings = model.encode(captions, convert_to_tensor=True)

    cos_sim = util.pytorch_cos_sim(query_embedding, caption_embeddings)

    top_indices = cos_sim.topk(3).indices.squeeze(0).tolist()
    top_captions = [captions[i] for i in top_indices]

    data.at[index, 'top_captions'] = ','.join(top_captions)

data['caption_match'] = data.apply(lambda x: x['image_caption'] in x['top_captions'].split(','), axis=1)

accuracy = data['caption_match'].mean()
print(f'Caption Match Accuracy: {accuracy:.4f}')

100%|██████████| 500/500 [01:13<00:00,  6.81it/s]

Caption Match Accuracy: 0.4600


### Test CLIP

In [ ]:
import tqdm
data['image_path_label'] = data.groupby('paper_id')['image_path'].transform(lambda x: ','.join(x))

from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

for index, row in tqdm.tqdm(data.iterrows(), total=len(data)):
    query = row['query']
    image_path_label = row['image_path_label']

    image_paths = image_path_label.split(',')

    images = []
    for image_path in image_paths:
        image = Image.open(f'data/train_images/train_images/{image_path}').convert("RGB")
        images.append(image)

    inputs = processor(text=[query]*len(images), images=images, return_tensors="pt", padding=True)
    outputs = model(**inputs)

    text_embeddings = outputs.text_embeds
    image_embeddings = outputs.image_embeds

    cos_sim = torch.nn.functional.cosine_similarity(text_embeddings, image_embeddings)

    top_indices = torch.topk(cos_sim, 3).indices.tolist()
    top_image_paths = [image_paths[i] for i in top_indices]

    data.at[index, 'top_image_paths'] = ','.join(top_image_paths)

data['image_path_match'] = data.apply(lambda x: x['image_path'] in x['top_image_paths'].split(','), axis=1)

accuracy = data['image_path_match'].mean()
print(f'Image Path Match Accuracy: {accuracy:.4f}')

100%|██████████| 500/500 [06:05<00:00,  1.37it/s]

Image Path Match Accuracy: 0.7900


### Test CrossEncoder

In [ ]:
from sentence_transformers import CrossEncoder
import tqdm
import torch

data['image_caption_label'] = data.groupby('paper_id')['image_caption'].transform(lambda x: ','.join(x))

reranker = CrossEncoder(
    "BAAI/bge-reranker-large",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

top_caption_results = []

for index, row in tqdm.tqdm(data.iterrows(), total=len(data)):
    query = row['query']
    caption_label = row['image_caption_label']

    captions = caption_label.split(',')

    pairs = [[query, cap] for cap in captions]

    scores = reranker.predict(pairs)

    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:3]

    top_captions = [captions[i] for i in top_indices]
    top_caption_results.append(",".join(top_captions))

data['top_captions'] = top_caption_results


data['caption_match'] = data.apply(
    lambda x: x['image_caption'] in x['top_captions'].split(','),
    axis=1
)

accuracy = data['caption_match'].mean()
print(f'Caption Match Accuracy (CrossEncoder): {accuracy:.4f}')


### Dataset (For fine-tune CLIP)

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

class ImageSearchDataset(Dataset):
    def __init__(self, json_path, processor, max_len=77, image_size=224):
        self.data = pd.read_json(json_path, lines=True)
        self.processor = processor
        self.max_len = max_len
        self.image_size = image_size

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        query = row["query"]
        image_path = row["image_path"]
        paper_id = row["paper_id"]

        image = Image.open(f'data/train_images/train_images/{image_path}').convert("RGB")

        inputs = self.processor(
            text=query,
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_len
        )
        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "pixel_values": inputs["pixel_values"].squeeze(0),
            "paper_id": paper_id
        }

### Fine-tune CLIP

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

dataset = ImageSearchDataset("data/train.jsonl", processor)
dataloader = DataLoader(dataset, batch_size=15)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)
temperature = 0.07

for epoch in range(10):
    model.train()
    epoch_loss = 0.0

    for batch in tqdm(dataloader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)

        paper_ids = batch["paper_id"]
        paper_ids = torch.tensor(
            [hash(pid) for pid in paper_ids],
            device=device
        )

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_loss=False
        )


        text_emb = F.normalize(outputs.text_embeds, dim=-1)
        image_emb = F.normalize(outputs.image_embeds, dim=-1)

        logits = (text_emb @ image_emb.T) / temperature  # [B, B]
        B = logits.size(0)

        # ======== mask construction ========
        same_paper = paper_ids[:, None] == paper_ids[None, :]   # [B, B]
        eye = torch.eye(B, dtype=torch.bool, device=device)

        positive_mask = eye                              # 只允許對角線
        negative_mask = same_paper & ~eye                # 同 paper 非對角線
        valid_mask = positive_mask | negative_mask       # 其餘忽略


        masked_logits = logits.masked_fill(~valid_mask, -1e9)

        labels = torch.arange(B, device=device)


        loss_t2i = F.cross_entropy(masked_logits, labels)

        loss_i2t = F.cross_entropy(masked_logits.T, labels)

        loss = 0.5 * (loss_t2i + loss_i2t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {epoch_loss / len(dataloader):.4f}")

100%|██████████| 549/549 [03:41<00:00,  2.47it/s]


Epoch 1, Loss: 0.9999


100%|██████████| 549/549 [03:42<00:00,  2.47it/s]


Epoch 2, Loss: 0.4855


100%|██████████| 549/549 [03:43<00:00,  2.46it/s]


...

Epoch 8, Loss: 0.0358


100%|██████████| 549/549 [03:40<00:00,  2.49it/s]


Epoch 9, Loss: 0.0296


100%|██████████| 549/549 [03:38<00:00,  2.51it/s]

Epoch 10, Loss: 0.0302


In [ ]:
model.save_pretrained("fine_tuned_clip")
processor.save_pretrained("fine_tuned_clip")

[]

### Submission

In [ ]:
import pandas as pd
test_query = pd.read_json('data/test.jsonl', lines=True)
test_image = pd.read_json('data/test_images.jsonl', lines=True)

In [ ]:
import easyocr
from PIL import Image
import numpy as np

reader = easyocr.Reader(['en'], gpu=True)

def extract_text_from_image(image_path, lang='en'):
    image = Image.open(image_path).convert('L')

    max_size = 1024
    if max(image.size) > max_size:
        ratio = max_size / max(image.size)
        new_size = (int(image.size[0] * ratio), int(image.size[1] * ratio))
        image = image.resize(new_size, Image.Resampling.LANCZOS)

    image_np = np.array(image)

    results = reader.readtext(image_np, detail=1, min_size=10, text_threshold=0.7, low_text=0.4)

    extracted_text = ' '.join([result[1] for result in results if result[2] > 0.5])

    return extracted_text

test_image['extracted_text'] = test_image['image_path'].apply(lambda x: extract_text_from_image(f'data/test_images/test_images/{x}'))

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [ ]:
test_image_grouped = test_image.groupby('paper_id').agg(lambda x: list(x)).reset_index()

test_image_dict = test_image_grouped.set_index('paper_id').to_dict(orient='index')

test_query['image_data'] = test_query['paper_id'].map(test_image_dict)

### Text to Text Submission

In [ ]:
submission_results = []

for index, row in tqdm.tqdm(test_query.iterrows(), total=len(test_query)):
    query = row['query']
    image_data_for_paper = row['image_data']

    query_embedding = model.encode(query, convert_to_tensor=True)

    image_scores_for_paper = []

    for i in range(len(image_data_for_paper['image_id'])):

        image_id = image_data_for_paper['image_id'][i]

        captions_list_for_this_image = image_data_for_paper['image_caption'][i]

        if isinstance(captions_list_for_this_image, str):
            captions_list_for_this_image = [captions_list_for_this_image]
        elif not isinstance(captions_list_for_this_image, list) or not all(isinstance(c, str) for c in captions_list_for_this_image):
            continue

        if not captions_list_for_this_image:
            continue

        caption_embeddings = model.encode(captions_list_for_this_image, convert_to_tensor=True)

        cos_sim = util.pytorch_cos_sim(query_embedding, caption_embeddings)

        max_sim_score_for_image = cos_sim.max().item()
        image_scores_for_paper.append((max_sim_score_for_image, image_id))

    image_scores_for_paper.sort(key=lambda x: x[0], reverse=True)

    top_image_ids = []
    seen_image_ids = set()
    for score, img_id in image_scores_for_paper:
        if img_id not in seen_image_ids:
            top_image_ids.append(str(img_id))
            seen_image_ids.add(img_id)
        if len(top_image_ids) == 3:
            break

    while len(top_image_ids) < 3:
        top_image_ids.append("")

    submission_results.append(f"{index+1},{' '.join(top_image_ids[:3])}")

with open('submission.csv', 'w') as f:
    f.write("id,image_id\n")
    f.write("\n".join(submission_results))

print("Submission file generated: submission.csv")

### Cross-encoder + OCR Submission

In [ ]:
import pandas as pd
import tqdm
import torch
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-large",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

submission_results = []

for index, row in tqdm.tqdm(test_query.iterrows(), total=len(test_query)):
    query = row['query']
    
    image_data_for_paper = row['image_data']

    image_scores_for_paper = []

    for i in range(len(image_data_for_paper['image_id'])):

        image_id = image_data_for_paper['image_id'][i]

        captions_list_for_this_image = image_data_for_paper['image_caption'][i]
        ocr_list_for_this_image = image_data_for_paper['extracted_text'][i]

        combine = f'Caption {captions_list_for_this_image} OCR {ocr_list_for_this_image}'

        if isinstance(combine, str):
            combine = [combine]
        elif not isinstance(combine, list) or not all(isinstance(c, str) for c in combine):
            continue

        if not combine:
            continue

        pairs = [[query, cap] for cap in combine]

        scores = reranker.predict(pairs)

        max_sim_score_for_image = max(scores)
        image_scores_for_paper.append((max_sim_score_for_image, image_id))

    image_scores_for_paper.sort(key=lambda x: x[0], reverse=True)

    top_image_ids = []
    seen_image_ids = set()
    for score, img_id in image_scores_for_paper:
        if img_id not in seen_image_ids:
            top_image_ids.append(str(img_id))
            seen_image_ids.add(img_id)
        if len(top_image_ids) == 3:
            break

    while len(top_image_ids) < 3:
        top_image_ids.append("")

    submission_results.append(f"{index+1},{' '.join(top_image_ids[:3])}")

with open('submission.csv', 'w') as f:
    f.write("id,image_id\n")
    f.write("\n".join(submission_results))

print("Submission file generated: submission.csv")

### Test Dual-Encoder

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DualEncoderModel(nn.Module):
    def __init__(
        self,
        image_encoder,
        text_encoder,
        image_dim: int,
        text_dim: int,
        embed_dim: int = 256
    ):
        super().__init__()

        self.image_encoder = image_encoder
        self.text_encoder = text_encoder

        for param in self.image_encoder.parameters():
            param.requires_grad = False
        for param in self.text_encoder.parameters():
            param.requires_grad = False

        self.image_proj = nn.Linear(image_dim, embed_dim)
        self.text_proj = nn.Linear(text_dim, embed_dim)

    def forward(self, image_inputs, text_inputs):
        image_outputs = self.image_encoder(**image_inputs) \
            if isinstance(image_inputs, dict) else self.image_encoder(image_inputs)

        if hasattr(image_outputs, "pooler_output"):
            image_feat = image_outputs.pooler_output
        else:
            image_feat = image_outputs.last_hidden_state.mean(dim=1)

        text_outputs = self.text_encoder(**text_inputs)

        if hasattr(text_outputs, "pooler_output"):
            text_feat = text_outputs.pooler_output
        else:
            text_feat = text_outputs.last_hidden_state[:, 0]  # CLS

        image_emb = self.image_proj(image_feat)
        image_emb = F.normalize(image_emb, dim=-1)

        text_emb = self.text_proj(text_feat)
        text_emb = F.normalize(text_emb, dim=-1)

        return image_emb, text_emb

In [ ]:
class DualEncoderDataset(torch.utils.data.Dataset):
    def __init__(self, data, image_processor, text_model):
        self.data = data
        self.image_processor = image_processor
        self.text_model = text_model

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        query = row["query"]
        image_path = row["image_path"]
        paper_id = row["paper_id"]

        image = Image.open(f'data/train_images/train_images/{image_path}').convert("RGB")
        image_inputs = self.image_processor(images=image, return_tensors="pt")
        for k, v in image_inputs.items():
            image_inputs[k] = v.squeeze(0)

        text_inputs = self.text_model.tokenize([query])
        for k, v in text_inputs.items():
            text_inputs[k] = v.squeeze(0)

        return {
            "image_inputs": image_inputs,
            "text_inputs": text_inputs,
            "paper_id": paper_id
        }

In [ ]:
# 加載數據
data = pd.read_json('data/train.jsonl', lines=True)

from transformers import ConvNextModel, ConvNextImageProcessor
from sentence_transformers import SentenceTransformer

image_processor = ConvNextImageProcessor.from_pretrained("microsoft/convnext-tiny-224")
image_encoder = ConvNextModel.from_pretrained("microsoft/convnext-tiny-224")
text_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# 創建數據集
dataset = DualEncoderDataset(data, image_processor, text_model)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=True)

# 創建模型
image_dim = image_encoder.config.hidden_size  # 假設是 ConvNeXt 的 hidden_size
text_dim = text_model.get_sentence_embedding_dimension()
model = DualEncoderModel(image_encoder, text_model, image_dim, text_dim, embed_dim=256).to(device)

# 定義優化器，只優化 projection layers
optimizer = torch.optim.AdamW([
    {'params': model.image_proj.parameters()},
    {'params': model.text_proj.parameters()}
], lr=1e-3)

temperature = 0.07

# 訓練循環
for epoch in range(10):
    model.train()
    epoch_loss = 0.0

    for batch in tqdm(dataloader):
        image_inputs = {k: v.to(device) for k, v in batch["image_inputs"].items()}
        text_inputs = {k: v.to(device) for k, v in batch["text_inputs"].items()}
        paper_ids = batch["paper_id"]

        # 前向傳播
        image_emb, text_emb = model(image_inputs, text_inputs)

        # 計算相似度
        logits = (text_emb @ image_emb.T) / temperature  # [B, B]

        # 構造 mask
        paper_ids = torch.tensor([hash(pid) for pid in paper_ids], device=device)
        same_paper = paper_ids[:, None] == paper_ids[None, :]
        eye = torch.eye(len(paper_ids), dtype=torch.bool, device=device)
        positive_mask = eye
        negative_mask = same_paper & ~eye
        valid_mask = positive_mask | negative_mask

        masked_logits = logits.masked_fill(~valid_mask, -1e9)
        labels = torch.arange(len(paper_ids), device=device)

        loss_t2i = F.cross_entropy(masked_logits, labels)
        loss_i2t = F.cross_entropy(masked_logits.T, labels)
        loss = 0.5 * (loss_t2i + loss_i2t)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {epoch_loss / len(dataloader):.4f}")

# 保存模型
torch.save(model.state_dict(), "dual_encoder_fine_tuned.pth")

In [ ]:
# load fine-tuned model
model.load_state_dict(torch.load("dual_encoder_fine_tuned.pth"))

model.eval()

submission_results = []

for index, row in tqdm.tqdm(test_query.iterrows(), total=len(test_query)):
    query = row['query']
    image_data_for_paper = row['image_data']

    image_scores_for_paper = []

    for i in range(len(image_data_for_paper['image_id'])):
        image_id = image_data_for_paper['image_id'][i]
        image_path = image_data_for_paper['image_path'][i]

        image = Image.open(f'data/test_images/test_images/{image_path}').convert("RGB")

        image_embedding, query_embedding = model.encode(image, query, convert_to_tensor=True)

        cos_sim = util.pytorch_cos_sim(query_embedding, image_embedding)

        max_sim_score_for_image = cos_sim.max().item()
        image_scores_for_paper.append((max_sim_score_for_image, image_id))

    image_scores_for_paper.sort(key=lambda x: x[0], reverse=True)

    top_image_ids = []
    seen_image_ids = set()
    for score, img_id in image_scores_for_paper:
        if img_id not in seen_image_ids:
            top_image_ids.append(str(img_id))
            seen_image_ids.add(img_id)
        if len(top_image_ids) == 3:
            break

    while len(top_image_ids) < 3:
        top_image_ids.append("")

    submission_results.append(f"{index+1},{' '.join(top_image_ids[:3])}")

with open('submission.csv', 'w') as f:
    f.write("id,image_id\n")
    f.write("\n".join(submission_results))

print("Submission file generated: submission.csv")